# 📓 Exemplo 06: Extrair os dados no Google Earth Engine

**Missão 5** · [![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/change_climate/blob/main/code/notebooks_exemplo/06_extrair_dados_gee.ipynb)

> **Pergunta:** de onde vêm os dados, e como buscá-los para qualquer lugar do Brasil?

### 🧭 Como usar este notebook
1. 💾 **Faça uma cópia:** menu **Arquivo → Salvar uma cópia no Drive**. Mova a cópia para a pasta
   `mudanca_climatica_br/notebooks/`. Assim você pode mudar tudo sem estragar o original.
2. ▶️ **Rode uma célula por vez** (botão ▶ ou **Shift + Enter**) e **leia os comentários** (as linhas com `#`).
3. 🧪 Os exemplos usam **Santa Helena-PR** e **Curitiba-PR**.
4. ✍️ No fim, a seção **Sua vez** pede para você **replicar o método para as capitais do Brasil**.

> ⚠️ **Antes:** você precisa de um **projeto registrado no Earth Engine** (Missão 5, semana 1). Anote o
> **ID** do projeto e coloque-o na variável `PROJETO` abaixo.

In [ ]:
!pip install earthengine-api

In [ ]:
import ee
import pandas as pd

PROJETO = '___'          # <- o ID do SEU projeto no Earth Engine

ee.Authenticate()        # abre a janela de login (siga as instruções)
ee.Initialize(project=PROJETO)
print('Conectada ao Earth Engine!')

## 1. Conhecendo a base do Xavier

In [ ]:
ASSET_XAVIER = 'projects/ee-alexandrexavier/assets/BR-DWGD'
colecao = ee.ImageCollection(ASSET_XAVIER)
primeira_imagem = colecao.first()

print('Variáveis (bandas):', primeira_imagem.bandNames().getInfo())
print('Primeiro dia:', ee.Date(primeira_imagem.get('system:time_start')).format('YYYY-MM-dd').getInfo())
print('Número de dias:', colecao.size().getInfo())

## 2. As funções de extração (leia os comentários)

In [ ]:
PRIMEIRO_ANO = 1961
ULTIMO_ANO = 2025


def converter_para_unidades_fisicas(imagem):
    """valor real = valor guardado * MULT + ADD (MULT e ADD vêm nas propriedades de cada imagem)."""
    tmax = imagem.select('Tmax')
    tmax = tmax.multiply(ee.Number(imagem.get('BAND_Tmax_MULT')))
    tmax = tmax.add(ee.Number(imagem.get('BAND_Tmax_ADD')))

    tmin = imagem.select('Tmin')
    tmin = tmin.multiply(ee.Number(imagem.get('BAND_Tmin_MULT')))
    tmin = tmin.add(ee.Number(imagem.get('BAND_Tmin_ADD')))

    chuva = imagem.select('pr')
    chuva = chuva.multiply(ee.Number(imagem.get('BAND_pr_MULT')))
    chuva = chuva.add(ee.Number(imagem.get('BAND_pr_ADD')))

    convertida = ee.Image.cat([tmax, tmin, chuva]).rename(['Tmax', 'Tmin', 'chuva'])
    convertida = convertida.copyProperties(imagem, ['system:time_start'])
    return ee.Image(convertida)


def baixar_dias(latitude, longitude, ano_inicio, ano_fim):
    """Valores DIÁRIOS de um ponto, de ano_inicio a ano_fim. Uma linha por dia."""
    ponto = ee.Geometry.Point([longitude, latitude])          # atenção: longitude primeiro!
    colecao = ee.ImageCollection(ASSET_XAVIER)
    colecao = colecao.filterDate(f'{ano_inicio}-01-01', f'{ano_fim + 1}-01-01')
    colecao = colecao.map(converter_para_unidades_fisicas)

    linhas = colecao.getRegion(ponto, scale=1000).getInfo()   # 1ª linha = cabeçalho
    tabela = pd.DataFrame(linhas[1:], columns=linhas[0])
    tabela['data'] = pd.to_datetime(tabela['time'], unit='ms')
    return tabela[['data', 'Tmax', 'Tmin', 'chuva']]


def serie_mensal_do_ponto(latitude, longitude):
    """Baixa de 10 em 10 anos e agrega em meses (temperatura = média; chuva = soma)."""
    pedacos = []
    for ano_inicio in range(PRIMEIRO_ANO, ULTIMO_ANO + 1, 10):
        ano_fim = min(ano_inicio + 9, ULTIMO_ANO)
        print('   baixando', ano_inicio, 'a', ano_fim)
        pedacos.append(baixar_dias(latitude, longitude, ano_inicio, ano_fim))

    diaria = pd.concat(pedacos).sort_values('data').set_index('data')
    diaria['tmed'] = (diaria['Tmax'] + diaria['Tmin']) / 2

    dias_no_mes = diaria['tmed'].resample('MS').size()
    mensal = pd.DataFrame()
    mensal['tmed'] = diaria['tmed'].resample('MS').mean()
    mensal['chuva'] = diaria['chuva'].resample('MS').sum()

    # Meses com algum dia faltando ficam vazios
    mensal.loc[diaria['tmed'].resample('MS').count() < dias_no_mes, 'tmed'] = float('nan')
    mensal.loc[diaria['chuva'].resample('MS').count() < dias_no_mes, 'chuva'] = float('nan')

    mensal.index.name = 'data'
    return mensal.round(2)

## 3. Extrair Santa Helena e conferir com o arquivo do GitHub

In [ ]:
# Endereço da pasta de dados do projeto no GitHub (arquivos "raw" = texto puro).
# O pandas consegue ler um CSV direto de um endereço da internet.
URL_DADOS = 'https://raw.githubusercontent.com/fcoliveira-utfpr/change_climate/main/code/dados'

In [ ]:
serie_minha = serie_mensal_do_ponto(-24.86, -54.33)

serie_github = pd.read_csv(f'{URL_DADOS}/santa_helena_mensal.csv', parse_dates=['data'], index_col='data')

print('Maior diferença de temperatura (°C):', (serie_minha['tmed'] - serie_github['tmed']).abs().max())
print('Maior diferença de chuva (mm):', (serie_minha['chuva'] - serie_github['chuva']).abs().max())

> ✅ As diferenças devem ser **zero**: você reproduziu os dados da pesquisa.

## ✍️ Sua vez: as 27 capitais
O exemplo extraiu **um** lugar. Agora você **repete** para cada capital.

**1. Monte a tabela das capitais.** Para cada capital, abra o [Google Maps](https://maps.google.com), clique
com o **botão direito** no centro da cidade e copie as coordenadas (o primeiro número é a latitude, o segundo
a longitude). Complete a tabela na célula abaixo.

In [ ]:
# Conecta o Drive, onde os dados vão ser salvos
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mudanca_climatica_br'

# Cada linha é uma capital. As duas primeiras são exemplos; complete as 27.
capitais = pd.DataFrame([
    {'uf': 'PR', 'capital': 'Curitiba', 'regiao': 'Sul', 'lat': -25.43, 'lon': -49.27},
    {'uf': 'DF', 'capital': 'Brasília', 'regiao': 'Centro-Oeste', 'lat': -15.79, 'lon': -47.88},
    # {'uf': '__', 'capital': '___', 'regiao': '___', 'lat': ___, 'lon': ___},
])

print('Número de capitais:', len(capitais))       # devem ser 27
capitais.to_csv(f'{BASE}/dados/capitais_27.csv', index=False)
capitais

**2. Repita à mão para as 3 primeiras.** Troque só as três linhas de cima e rode de novo para cada capital.
Antes, crie a pasta `capitais` dentro de `dados/` no Drive.

In [ ]:
# Troque estas três linhas para cada capital
UF = 'pr'                  # sigla em minúsculas: vira o nome do arquivo
LATITUDE = -25.43
LONGITUDE = -49.27

# O resto é igual para todas
serie = serie_mensal_do_ponto(LATITUDE, LONGITUDE)
serie.to_csv(f'{BASE}/dados/capitais/{UF}.csv')
print(UF, '-> meses vazios:', serie['tmed'].isna().sum() + serie['chuva'].isna().sum())

> ⚠️ **Se a série vier toda vazia**, o ponto caiu **no mar** (a grade do Xavier só cobre o continente). Mova a
> coordenada um pouco para o **interior**, anote a mudança na tabela e extraia de novo.

> 🧠 **Confira:** a sua série de PR (Curitiba) é igual ao `curitiba_mensal.csv` do projeto? Use a mesma
> conferência que fizemos para Santa Helena.

**3. Repita para todas, com o `for`.** A célula abaixo faz a **mesma coisa** para cada linha da tabela. É o
mesmo código da célula anterior, repetido automaticamente (você vai aprender o `for` de verdade na Missão 8).
Leva cerca de 1 minuto por capital.

In [ ]:
# 🔧 CÉLULA PRONTA: repete a extração para cada capital da tabela
for indice, capital in capitais.iterrows():
    uf = capital['uf'].lower()
    print('Capital:', capital['capital'])

    serie = serie_mensal_do_ponto(capital['lat'], capital['lon'])
    serie.to_csv(f'{BASE}/dados/capitais/{uf}.csv')

    print('   meses vazios:', serie['tmed'].isna().sum() + serie['chuva'].isna().sum())